### Feature Engineering

#### Creating new features in our dataset for modelling

#### Importing Packages and Data

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
project_root = Path(".").resolve().parent
input_path = project_root / "data" / "processed" / "london_clean.parquet"
output_path = project_root / "data" / "processed" / "london_features.parquet"

df = pd.read_parquet(input_path)

In [4]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS
...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE


#### Features 1

In [5]:
# Postcode District: Outward code (area + district)
df["postcode_district"] = df["postcode"].str.split().str[0]

df["postcode_district"]

0          E3
1         E17
2         NW4
3          E2
4         E14
         ... 
72587     E10
72588    RM13
72589     RM3
72590     E18
72591    RM13
Name: postcode_district, Length: 72592, dtype: object

In [6]:
# Property Type: Flat, Terraced, Semi-Detached or Detached
type_order = {"F": 1, "T": 2, "S": 3, "D": 4}
df["property_type_ordinal"] = df["property_type"].map(type_order).astype("int16")

df["property_type_ordinal"]

0        1
1        1
2        3
3        1
4        1
        ..
72587    1
72588    3
72589    2
72590    3
72591    2
Name: property_type_ordinal, Length: 72592, dtype: int16

In [7]:
# Sales count in District: How many properties are sold in the district
district_counts = df["postcode_district"].value_counts()
df["district_sales_count"] = df["postcode_district"].map(district_counts).astype("int16")

df["district_sales_count"]

0         530
1        1046
2         212
3         330
4         879
         ... 
72587     450
72588     314
72589     402
72590     226
72591     314
Name: district_sales_count, Length: 72592, dtype: int16

In [8]:
# Log Price
df["log_price"] = np.log10(df["price"])

df["log_price"]

0        5.332438
1        5.542825
2        5.732394
3        5.662758
4        5.423246
           ...   
72587    5.595276
72588    5.588832
72589    5.553883
72590    6.039414
72591    5.526339
Name: log_price, Length: 72592, dtype: float64

In [9]:
print(f"Unique Postcode Districts: {df['postcode_district'].nunique()}")
print(f"District Sales Count Range: {df['district_sales_count'].min()} to {df['district_sales_count'].max()}")
print(f"Property type ordinal counts:\n{df['property_type_ordinal'].value_counts().sort_index()}")
print(f"Log price skewness: {df['log_price'].skew():.2f}")

Unique Postcode Districts: 273
District Sales Count Range: 1 to 1339
Property type ordinal counts:
property_type_ordinal
1    36826
2    20901
3    11427
4     3438
Name: count, dtype: int64
Log price skewness: 0.69


In [10]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E3,1,530,5.332438
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E17,1,1046,5.542825
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW4,3,212,5.732394
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E2,1,330,5.662758
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E14,1,879,5.423246
...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E10,1,450,5.595276
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM13,3,314,5.588832
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM3,2,402,5.553883
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E18,3,226,6.039414


#### Features 2

In [11]:
# The london_postcodes.csv file (from doogal.co.uk) maps every London postcode to its latitude & longitude, London zone, and distance calculated to nearest station.

postcodes_path = project_root / "data" / "raw" / "london_postcodes.csv"

# The file is +250MB, loading the columns we need:
pc_df = pd.read_csv(
    postcodes_path,
    usecols=["Postcode", "In Use?", "Latitude", "Longitude", "London zone", "Distance to station"]
)

In [12]:
pc_df

,Postcode,In Use?,Latitude,Longitude,London zone,Distance to station
0,BR1 1AA,Yes,51.401545,0.015440,5.0,0.217100
1,BR1 1AB,Yes,51.406333,0.015234,4.0,0.252793
2,BR1 1AD,No,51.400057,0.016741,5.0,0.042797
3,BR1 1AE,Yes,51.404543,0.014221,4.0,0.462186
4,BR1 1AF,Yes,51.401391,0.014973,5.0,0.226316
...,...,...,...,...,...,...
332303,WD3 8UX,Yes,51.624762,-0.494021,7.0,2.226450
332304,WD3 8UZ,Yes,51.626957,-0.494122,7.0,2.047580
332305,WD3 8XD,Yes,51.628578,-0.499183,7.0,2.189930
332306,WD6 2RN,Yes,51.643273,-0.255930,6.0,1.963070


In [13]:
# Filtering to active postcodes only
pc = pc_df[pc_df['In Use?'] == 'Yes']

pc

,Postcode,In Use?,Latitude,Longitude,London zone,Distance to station
0,BR1 1AA,Yes,51.401545,0.015440,5.0,0.217100
1,BR1 1AB,Yes,51.406333,0.015234,4.0,0.252793
3,BR1 1AE,Yes,51.404543,0.014221,4.0,0.462186
4,BR1 1AF,Yes,51.401391,0.014973,5.0,0.226316
5,BR1 1AG,Yes,51.401391,0.014973,5.0,0.226316
...,...,...,...,...,...,...
332303,WD3 8UX,Yes,51.624762,-0.494021,7.0,2.226450
332304,WD3 8UZ,Yes,51.626957,-0.494122,7.0,2.047580
332305,WD3 8XD,Yes,51.628578,-0.499183,7.0,2.189930
332306,WD6 2RN,Yes,51.643273,-0.255930,6.0,1.963070


In [14]:
# Renaming Columns
pc = pc.rename(columns={
    'Postcode': 'postcode',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'London zone': 'london_zone',
    'Distance to station': 'distance_to_station_km'})

pc

,postcode,In Use?,latitude,longitude,london_zone,distance_to_station_km
0,BR1 1AA,Yes,51.401545,0.015440,5.0,0.217100
1,BR1 1AB,Yes,51.406333,0.015234,4.0,0.252793
3,BR1 1AE,Yes,51.404543,0.014221,4.0,0.462186
4,BR1 1AF,Yes,51.401391,0.014973,5.0,0.226316
5,BR1 1AG,Yes,51.401391,0.014973,5.0,0.226316
...,...,...,...,...,...,...
332303,WD3 8UX,Yes,51.624762,-0.494021,7.0,2.226450
332304,WD3 8UZ,Yes,51.626957,-0.494122,7.0,2.047580
332305,WD3 8XD,Yes,51.628578,-0.499183,7.0,2.189930
332306,WD6 2RN,Yes,51.643273,-0.255930,6.0,1.963070


In [15]:
pc = pc.drop(columns=['In Use?'])

pc

,postcode,latitude,longitude,london_zone,distance_to_station_km
0,BR1 1AA,51.401545,0.015440,5.0,0.217100
1,BR1 1AB,51.406333,0.015234,4.0,0.252793
3,BR1 1AE,51.404543,0.014221,4.0,0.462186
4,BR1 1AF,51.401391,0.014973,5.0,0.226316
5,BR1 1AG,51.401391,0.014973,5.0,0.226316
...,...,...,...,...,...
332303,WD3 8UX,51.624762,-0.494021,7.0,2.226450
332304,WD3 8UZ,51.626957,-0.494122,7.0,2.047580
332305,WD3 8XD,51.628578,-0.499183,7.0,2.189930
332306,WD6 2RN,51.643273,-0.255930,6.0,1.963070


##### Merging the Property Sales and Postcodes datasets

In [16]:
df = df.merge(pc, on='postcode', how='left')

df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price,latitude,longitude,london_zone,distance_to_station_km
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E3,1,530,5.332438,51.530123,-0.016067,2.0,0.414205
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E17,1,1046,5.542825,51.582287,-0.028159,3.0,0.309116
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW4,3,212,5.732394,51.577269,-0.230011,3.0,0.671918
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E2,1,330,5.662758,51.534910,-0.074545,1.0,0.385880
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E14,1,879,5.423246,51.506823,-0.005480,2.0,0.165213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E10,1,450,5.595276,51.567063,-0.030880,3.0,0.403633
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM13,3,314,5.588832,51.525933,0.219288,6.0,2.195610
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM3,2,402,5.553883,51.602708,0.205338,6.0,2.213750
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E18,3,226,6.039414,51.599639,0.014872,4.0,1.273380


In [41]:
df.dtypes

price                              int64
date_of_transfer          datetime64[ns]
postcode                          object
property_type                   category
old_new                         category
duration                        category
district                        category
postcode_district                 object
property_type_ordinal              int16
district_sales_count               int16
log_price                        float64
latitude                         float64
longitude                        float64
london_zone                      float64
distance_to_station_km           float64
is_central                         int16
dtype: object

In [42]:
df["london_zone"] = df["london_zone"].round().astype("int8")

In [19]:
# Missing Values
df[df["latitude"].isna()]

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price,latitude,longitude,london_zone,distance_to_station_km
273,1110000,2025-08-04,HA6 3AW,D,N,F,HILLINGDON,HA6,4,170,6.045323,NaN,NaN,NaN,NaN
756,2100000,2025-12-19,SW7 5PP,F,N,L,KENSINGTON AND CHELSEA,SW7,1,157,6.322219,NaN,NaN,NaN,NaN
9984,2000000,2025-12-19,SW7 1DS,F,N,L,CITY OF WESTMINSTER,SW7,1,157,6.301030,NaN,NaN,NaN,NaN
14922,395000,2025-03-27,SE1 8HD,F,N,L,SOUTHWARK,SE1,1,584,5.596597,NaN,NaN,NaN,NaN
15942,380000,2025-01-23,SE27 0AP,T,N,F,LAMBETH,SE27,2,240,5.579784,NaN,NaN,NaN,NaN
20026,3400000,2025-08-01,W1G 6HR,F,N,L,CITY OF WESTMINSTER,W1G,1,23,6.531479,NaN,NaN,NaN,NaN
21247,360000,2025-03-11,W12 9DU,T,N,L,HAMMERSMITH AND FULHAM,W12,2,348,5.556303,NaN,NaN,NaN,NaN
44646,318000,2025-02-21,HA8 5AH,F,N,L,HARROW,HA8,1,353,5.502427,NaN,NaN,NaN,NaN
47464,295000,2025-03-07,SE11 4HG,F,N,L,LAMBETH,SE11,1,189,5.469822,NaN,NaN,NaN,NaN
66904,540000,2025-03-27,RM12 4FY,T,N,F,HAVERING,RM12,2,448,5.732394,NaN,NaN,NaN,NaN


In [22]:
# Instead of dropping these rows, we will fill the missing values by using the close neighbours in the same postcode district. The unmatched properties will now have the values of a typical location of its neighborhood.

geo_cols = ["latitude", "longitude", "london_zone", "distance_to_station_km"]

for col in geo_cols:
    df[col] = df[col].fillna(
        df.groupby("postcode_district", observed=True)[col].transform("median")
    )



In [25]:
df["latitude"].isna().sum()

np.int64(0)

In [24]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price,latitude,longitude,london_zone,distance_to_station_km
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E3,1,530,5.332438,51.530123,-0.016067,2.0,0.414205
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E17,1,1046,5.542825,51.582287,-0.028159,3.0,0.309116
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW4,3,212,5.732394,51.577269,-0.230011,3.0,0.671918
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E2,1,330,5.662758,51.534910,-0.074545,1.0,0.385880
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E14,1,879,5.423246,51.506823,-0.005480,2.0,0.165213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E10,1,450,5.595276,51.567063,-0.030880,3.0,0.403633
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM13,3,314,5.588832,51.525933,0.219288,6.0,2.195610
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM3,2,402,5.553883,51.602708,0.205338,6.0,2.213750
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E18,3,226,6.039414,51.599639,0.014872,4.0,1.273380


In [43]:
# Central London feature
df["is_central"] = (df["london_zone"] <= 2).astype("int8")

df["is_central"]

0        1
1        0
2        0
3        1
4        1
        ..
72587    0
72588    0
72589    0
72590    0
72591    0
Name: is_central, Length: 72592, dtype: int8

In [30]:
print(f"Central London - Zone 1-2 properties: {df['is_central'].mean():.1%}")

print(df.groupby('is_central')['price'].median().apply(lambda x: f"£{x:,.0f}"))

Central London - Zone 1-2 properties: 28.4%
is_central
0    £500,000
1    £620,000
Name: price, dtype: object


In [33]:
print ("Median Price in London Zones")
print(df.groupby('london_zone')['price'].median().apply(lambda x: f"£{x:,.0f}"))

Median Price in London Zones
london_zone
1.0    £780,000
2.0    £599,625
3.0    £525,000
4.0    £490,000
5.0    £490,000
6.0    £475,000
7.0    £475,000
8.0    £310,000
Name: price, dtype: object


In [37]:
print("Median Prices by Distance to TFL Stations")
distance_bucket = pd.cut(
    df['distance_to_station_km'],
    bins=[0, 0.25, 0.5, 1.0, 2.0, 10],
    labels=['<0.25km', '0.25-0.5km', '0.5-1km', '1-2km', '>2km'],
)
print(df.groupby(distance_bucket, observed=True)['price'].median().apply(lambda x: f"£{x:,.0f}"))

Median Prices by Distance to TFL Stations
distance_to_station_km
<0.25km       £509,500
0.25-0.5km    £550,000
0.5-1km       £535,000
1-2km         £500,000
>2km          £470,000
Name: price, dtype: object


In [38]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_district,property_type_ordinal,district_sales_count,log_price,latitude,longitude,london_zone,distance_to_station_km,is_central
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E3,1,530,5.332438,51.530123,-0.016067,2.0,0.414205,1
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E17,1,1046,5.542825,51.582287,-0.028159,3.0,0.309116,0
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW4,3,212,5.732394,51.577269,-0.230011,3.0,0.671918,0
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E2,1,330,5.662758,51.534910,-0.074545,1.0,0.385880,1
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E14,1,879,5.423246,51.506823,-0.005480,2.0,0.165213,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E10,1,450,5.595276,51.567063,-0.030880,3.0,0.403633,0
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM13,3,314,5.588832,51.525933,0.219288,6.0,2.195610,0
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM3,2,402,5.553883,51.602708,0.205338,6.0,2.213750,0
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E18,3,226,6.039414,51.599639,0.014872,4.0,1.273380,0


In [44]:
df.dtypes

price                              int64
date_of_transfer          datetime64[ns]
postcode                          object
property_type                   category
old_new                         category
duration                        category
district                        category
postcode_district                 object
property_type_ordinal              int16
district_sales_count               int16
log_price                        float64
latitude                         float64
longitude                        float64
london_zone                         int8
distance_to_station_km           float64
is_central                          int8
dtype: object

In [39]:
# Saving the Feature Dataset
df.to_parquet(output_path)